In [6]:
import flordb as flor
import pandas as pd

# This is the "long" table of log records (one row per log event)
records = flor.dataframe("event")  # try a likely key you logged
records.tail(20)

,projid,tstamp,filename,event
0,lerobotwithflor,2026-02-25 13:40:39,record_with_flor.py,recording_start
1,lerobotwithflor,2026-02-25 13:40:39,record_with_flor.py,recording_end


In [8]:
import flordb as flor

df = flor.dataframe()    
df


,projid,tstamp,filename,run_name,dataset_root,robot_port,teleop_port,repo_id,num_episodes,episode_time_s,...,push_to_hub,task,event,started_utc,finished_utc,return_code,elapsed_s,success,output_contains_timeout,output_contains_read_failed
0,lerobotwithflor,2026-02-25 13:40:39,record_with_flor.py,so101_3cam_20260225_134040,C:\Users\Erick Duarte\git\lerobotwithflor\data...,COM6,COM5,BarbaricErick/so101_3cam_demo,10,180,...,False,Pick up the red cylinder and place it centered...,recording_start,2026-02-25T20:40:40Z,2026-02-25T20:48:52Z,0,492.4615879058838,True,False,False


In [10]:
import flordb as flor
flor.dataframe("event", "success", "elapsed_s")


,projid,tstamp,filename,event,success,elapsed_s
0,lerobotwithflor,2026-02-25 13:40:39,record_with_flor.py,recording_start,True,492.4615879058838
1,lerobotwithflor,2026-02-25 13:40:39,record_with_flor.py,recording_end,True,492.4615879058838


In [11]:
import os
import json
from pathlib import Path

root = Path("datasets")

runs = sorted(root.glob("so101_3cam_*"))
runs

[WindowsPath('datasets/so101_3cam_20260223_135159'),
 WindowsPath('datasets/so101_3cam_20260225_121810'),
 WindowsPath('datasets/so101_3cam_20260225_134040')]

In [19]:
manifest = runs[0] / "run_manifest.json"

with open(manifest) as f:
    data = json.load(f)

data.keys()

dict_keys(['tool', 'started_utc', 'run_name', 'dataset_root', 'host', 'config', 'command', 'finished_utc', 'return_code', 'elapsed_s', 'console_tail'])

In [20]:
from pathlib import Path

root = Path("datasets")
runs = sorted(root.glob("so101_3cam_*"))

for r in runs:
    print("\n", r.name)
    for f in r.iterdir():
        print("  ", f.name)


 so101_3cam_20260223_135159
   data
   meta
   run_manifest.json
   videos

 so101_3cam_20260225_121810
   data
   meta
   run_manifest.json
   videos

 so101_3cam_20260225_134040
   data
   meta
   run_manifest.json
   videos


In [21]:
ep_dir = runs[0] / "episodes"
len(list(ep_dir.glob("*")))

0

In [22]:
from pathlib import Path

root = Path("datasets")
runs = sorted(root.glob("so101_3cam_*"))

for r in runs:
    print("\n", r.name)
    data_dir = r / "data"
    eps = list(data_dir.glob("*"))
    print("  episodes:", len(eps))


 so101_3cam_20260223_135159
  episodes: 1

 so101_3cam_20260225_121810
  episodes: 1

 so101_3cam_20260225_134040
  episodes: 1


In [23]:
from pathlib import Path
import pandas as pd

# pick one run
run = Path("datasets/so101_3cam_20260225_134040")
parquet_path = run / "data" / "chunk-000" / "file-000.parquet"

df = pd.read_parquet(parquet_path)

df.head()


,action,observation.state,timestamp,frame_index,episode_index,index,task_index
0,"[-7.0343723, -99.74619, 94.6388, 67.800255, -1...","[-13.053613, -98.90202, 100.0, 69.37716, -0.70...",0.000000,0,0,0,0
1,"[-7.0343723, -99.74619, 94.6388, 67.800255, -1...","[-13.053613, -98.90202, 100.0, 69.37716, -0.70...",0.033333,1,0,1,0
2,"[-7.0343723, -99.74619, 94.6388, 67.800255, -1...","[-11.111111, -98.90202, 98.91697, 69.03114, -1...",0.066667,2,0,2,0
3,"[-7.0343723, -99.74619, 94.6388, 67.800255, -1...","[-8.624708, -99.49324, 98.736465, 68.94463, -1...",0.100000,3,0,3,0
4,"[-7.0343723, -99.74619, 94.6388, 67.800255, -1...","[-7.3815074, -99.324326, 98.736465, 68.94463, ...",0.133333,4,0,4,0


In [24]:
df.columns

Index(['action', 'observation.state', 'timestamp', 'frame_index',
       'episode_index', 'index', 'task_index'],
      dtype='object')

In [25]:
import numpy as np

# shapes of your vectors
a0 = np.array(df.loc[0, "action"], dtype=float)
s0 = np.array(df.loc[0, "observation.state"], dtype=float)
print("action dim:", a0.shape, "state dim:", s0.shape)

# how many episodes + how long each one is (steps)
print("episodes:", df["episode_index"].nunique())
steps_per_ep = df.groupby("episode_index")["index"].count()
display(steps_per_ep.describe())
display(steps_per_ep.head(20))

action dim: (6,) state dim: (6,)
episodes: 10


count     10.000000
mean     210.300000
std       41.593402
min      159.000000
25%      181.750000
50%      204.000000
75%      233.250000
max      299.000000
Name: index, dtype: float64

episode_index
0    174
1    189
2    219
3    222
4    237
5    159
6    184
7    239
8    181
9    299
Name: index, dtype: int64

In [26]:
import numpy as np
a0 = np.array(df.loc[0, "action"], dtype=float)
s0 = np.array(df.loc[0, "observation.state"], dtype=float)
print("action dim:", a0.shape, "state dim:", s0.shape)
print("episodes:", df["episode_index"].nunique())
print(df.groupby("episode_index")["index"].count().describe())


action dim: (6,) state dim: (6,)
episodes: 10
count     10.000000
mean     210.300000
std       41.593402
min      159.000000
25%      181.750000
50%      204.000000
75%      233.250000
max      299.000000
Name: index, dtype: float64


In [27]:
from pathlib import Path
run = Path("datasets/so101_3cam_20260225_134040")
print("video files:", len(list((run/"videos").glob("*"))))
print(sorted([p.name for p in (run/"videos").glob("*")])[:20])

video files: 3
['observation.images.front', 'observation.images.overhead', 'observation.images.wrist']


In [29]:
import pandas as pd
from pathlib import Path

roots = sorted(Path("datasets").glob("so101_3cam_*"))

dfs = []
episode_offset = 0

for r in roots:
    pq = list((r/"data").glob("**/*.parquet"))[0]
    df = pd.read_parquet(pq)

    # Shift episode indices so they don’t collide
    df["episode_index"] += episode_offset

    episode_offset = df["episode_index"].max() + 1
    dfs.append(df)

full = pd.concat(dfs, ignore_index=True)

print("Total episodes:", full["episode_index"].nunique())
print("Total steps:", len(full))

Total episodes: 30
Total steps: 7805


In [30]:
steps_per_ep = full.groupby("episode_index")["index"].count()
steps_per_ep.describe()

count     30.000000
mean     260.166667
std       54.281567
min      159.000000
25%      230.250000
50%      254.000000
75%      299.000000
max      362.000000
Name: index, dtype: float64